# Pincode Clustering - West Bengal

## 1. Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

## 2. Load raw data

In [ ]:
# low_memory = False prevents pandas from guessing data types column to column
# which can cause same column to have mixed types across chunks
df = pd.read_csv("clustering_data.csv", low_memory=False)
df.info()

In [ ]:
df.head()

In [ ]:
df.describe(include="all")

In [ ]:
null_counts_all = df.isnull().sum()
print(null_counts_all[null_counts_all > 0])

### Defining the task and metrics in consideration

**Task**:
1. Given task is to extract entries for West Bengal and filter out pincodes.
2. Preprocess to remove missing values, duplicates.
3. K-Means clustering on Longitude,Latitutude data to filter pincodes and identify district clusters
4. Visualise and draw meaningful inferences from clustering results. Provide insights about the geographical distribution and potential implications.
5. There is no particular issue to be singled on like finding optimal hub locations, cost,terrain, distribution etc so theres no optimal algorithm to consider nor optimal K or other hyperparameters as well.
   
   Keeping above points in mind the only things to take in consideration while clustering are 
- Proper preprocessing
- Objective and distance metrics considering real geographic data
* Additionally I compared K-Means with K-Medoids + FuzzyCMeans combined as one hybrid method which makes sense in geographical data 
   
**Data Metrics and Considerations**:

1. **Rows to consider:**
    1. Pincodes - We would need weighted K-Means since one Pincode could contain multiple offices. Further if a particular task was given we could give different weights to different office types and delivery as well to meet particular conditions.
    Clustering based on pincodes could also help analyse how pincodes are actually distributed in correlation to geographic distribution
    2. Post-offices (already given) -  Clustering based on this  would help find out office densities, assuming each office was made based on the population needs and no other fixed construction plan we can figure out population densities in each region as well.
    Again we can use weights to diff type of offices for particular problem solving.
2. **Data quality:**
   From basic checks on raw data the most noticeable thing is the number of null values in lat / lon data (like 1/15th of it)-> removing null values in subset is significant.
   1. **Type fixing** : From `df.info()`, lat lon are `objects`. Need to convert to floats first so `min()`,`max()`, `isnull()` make sense
   2. **Null checks** : After fixing data types, data type conversion errors + already NULL values can be filtered out 
   3. **Geographic boundary check**: Coordinates physically inside boundary
   4. **Duplicate removal**: In context of office rows its removing the same entry twice but in context of pincode its multiple offices per pincode
   5. **Normalisation need** -  degree of longitude is not the same physical distance as a degree of latitude. Normalising wont fix this - 
    ```
    For Bengal:  lat range = 5.7 deg,  lon range = 4.0 deg

    After min-max:  1 unit lat = 5.7 deg x 111 km/ deg = 632.7 km
                    1 unit lon = 4.0 deg x 102 km/ deg = 408.0 km

    That is a 55% distortion. East-west clusters are stretched relative to north-south.
    ```
3. **Distance metric**:
    1. Lat and lon are angular coordinates on a sphere, euclidean is on a flat plane. On looking around for proper distance metrics i found these two - 
        - Equirectangular approximation - Scale longitude differences by `cos(mean_lat)` before computing euclidean. This gives distance in degrees with correct aspect ratio 
        - Haversine formula - For larger regions with exact spherical distance.
    2. If we cluster at office level : Clustering is biased towards dense regions, not because there are more pincodes there but more office branches - which is an **Post staffing/ other metric decision, not a geographic property of pincode distribution**
    3. If we cluster at pincode level: Each pincode is one point and we would need a representative coordinate which could be
        - Head office coords (if there)
        - Mean of offices

**Selecting K**:
Based on domain facts - we can start with `state["DivisionName"].nunique()=28` as our upper bound.
Another guess we can take is `state["District"].nunique()=23`. Further based on more geographical data we can take different K guesses.
Silhouette and objective elbow measure compactness and nearness or in general any metric would optimise its specific metric and give clusters different meaning, we cant generalise them to choose K. I chose it to be mix of silhouette, elbow and geographical data.

**K-Means + other similar clustering methods**:
- K-Means clusters are spherical and it minimises within cluster sum squares of distances. Dense regions would be over segmented and sparse regions under because they contribute disproportionately to the objective.
- Hard cluster assignments since its not probabilistic clustering
- Since we properly preprocess the data, the chance of extreme outliers is very less given any state, still K-Means is significantly affected by that as well

Other adjacent clustering methods:
1. K-Medoids - We minimise physical distances (not square) and use actual data points as medoids instead of mean of coordinates - more robust to outliers and makes more sense with geographical data, as mean could lie in some physically infeasible region
2. FuzzyCMeans - soft assignment, assigning membership probabilities. Use K-Medoids cluster boundaries to initialise FCM membership matrix - so we get both the outlier robustness of medoids and the soft boundary analysis of FCM in one pipeline

### Choices made:
1. Cluster at Pincode level as asked using latitude and longitude
2. Use equirectangular approximation - scale longitude differences by `cos(mean_lat)` -> euclidean distance
3. Choose K from a set of values based on objective scores and geographical data
4. K-Medoids to get hard robust clusters, FCM initialised from medoid assignments to analyse boundary ambiguity

## 3. Preprocessing

In [ ]:
# Setting state name variable if required for testing elsewhere but would need to change all state-specific constraints elsewhere as well
state_name = "WEST BENGAL"
state = df[df["StateName"].str.upper() == 'WEST BENGAL'].copy()

# Basic stats
print(f"Total rows: {state.shape[0]}")
print(f"Unique pincodes: {state["Pincode"].nunique()}")
print(f"Unique districts: {state["District"].nunique()}")
print(f"Unique divisions: {state["DivisionName"].nunique()}")

In [ ]:
# Preprocssing in the order written above
# 1. Change data types of lat/lon to floats from objects
# set the data which cant be parsed to NaN with coerce
state["Latitude"] = pd.to_numeric(state["Latitude"],errors="coerce")
state["Longitude"] = pd.to_numeric(state["Longitude"],errors='coerce')

# 2. Count Null values and remove them - will now inlcude both unparseable + intial null vals
null_count = state.isnull().sum()
print(f"Null Count check : {null_count}")
# cooridnates are a physical fact - cant be imported and are our main clustering inputs and hence need to be dropped
n_before = len(state)
state = state.dropna(subset=['Latitude','Longitude'])
print(f"Dropped {n_before-len(state)} rows with missing coords")

In [ ]:
# 3. geographic boundary check 
# check current ranges 
print(f"Raw lat range : {state['Latitude'].min():.4f} to {state['Latitude'].max():.4f}")
print(f"Raw lon range : {state['Longitude'].min():.4f} to {state['Longitude'].max():.4f}")

# West bengal actual bounds : lat 21.5-27.2 , lon 85.5 - 89.9
################ change these values if running on diff state ########################
LAT_MIN, LAT_MAX = 21.0, 27.5
LON_MIN, LON_MAX = 85.5, 90.0
out_of_bounds = state[~(state['Latitude'].between(LAT_MIN, LAT_MAX) & state['Longitude'].between(LON_MIN, LON_MAX))]
print(f"Out of physical bounds rows : {len(out_of_bounds)}")

n_before = len(state)
state = state[state['Latitude'].between(LAT_MIN,LAT_MAX) & state['Longitude'].between(LON_MIN,LON_MAX)].copy()
print(f"Remove {n_before-len(state)} rows \nRemaining :{len(state)}")

In [ ]:
# Step 4: duplicate removal - exact row duplicates only
# multiple offices per pincode is not a duplicate
exact_dups = state.duplicated().sum()
print(f"Exact duplicate rows: {exact_dups}")
state = state.drop_duplicates()

offices_per_pin = state.groupby('Pincode').size()
print(f"Offices per pincode stats:")
print(offices_per_pin.describe())

### 3.1 Aggregate to pincode level

Clustering at office level biases toward dense regions (more office branches = more votes). Aggregating to one row per pincode fixes this.

Representative coordinate:
- Use Head Office coords if one exists for that pincode
- Otherwise use mean of all offices

In [ ]:
# to find appropriate head for each group of pincodes -> groupby pincode
def get_representative(group):
    head = group[group["OfficeType"].str.upper().str.contains("HO")]
    src = head.iloc[0] if len(head) > 0 else group.iloc[0] 
    return pd.Series({
        'Latitude' : src['Latitude'] if len(head)>0 else group['Latitude'].mean(),
        'Longitude': src['Longitude'] if len(head) > 0 else group['Longitude'].mean(),
        'OfficeName': src['OfficeName'],
        'District': src['District'],
        'DivisionName': src['DivisionName'],
        'RegionName': src['RegionName'],
        'n_offices': len(group)
    })

pincode_df = state.groupby('Pincode').apply(get_representative).reset_index()
print(pincode_df.head())

## 4. Distance metric

- Equirectangular approximation: scale longitude by `cos(mean_lat)` before euclidean.
Accurate to ~0.5% for regions under 1000km -- West Bengal (~600km tall) is within this.<br>
- Haversine would be exact but the difference at this scale is negligible.<br>
- Pre calc and keep a distance matirx for usage whenever needed in other methods to avoid recomputation each time

In [ ]:
mean_lat_rad = np.radians(pincode_df['Latitude'].mean())
COS_LAT = np.cos(mean_lat_rad)

print(f"1 deg lat : 111 km")
print(f"1 deg lon : {111*COS_LAT:.5f} km ")

In [ ]:
# create a 2D distance matrix for easy access and one time compute
def dist_matrix(X,C):
    diff = X[:,None,:] - C[None,:,:] # (n,2) - (k,2) -> (n,1,2) - (1,k,2) = (n,k,2) and along axis=2 lat nd lon are there
    # apply correction
    diff[:,:,1] *= COS_LAT
    # returns a 2D array (n,k) with distance of ith point from jth centroid at (i,j)
    return np.sqrt((diff**2).sum(axis=2))

ref = np.array([[1.0,2.0]])
# dist_matrix(ref,np.array([[3.0,4.0]]))

In [ ]:
# extract only lat and lon for clustering 
X = pincode_df[['Latitude', 'Longitude']].values.astype(float)
print(f"Data matrix : {X.shape}")
print(f"Lat range   : {X[:,0].min():.4f} to {X[:,0].max():.4f}")
print(f"Lon range   : {X[:,1].min():.4f} to {X[:,1].max():.4f}")

## 5. Pre-clustering visualisation

In [ ]:
# randomly generated colors list for plotting
COLORS = ['red','blue','green','orange','purple','brown','pink','gray','black','cyan','olive','teal','navy','coral','gold','lime','maroon','violet','salmon','indigo'] 

fig,ax = plt.subplots(1, 2, figsize=(12, 6))
fig.suptitle('West Bengal Pincodes - Before Clustering')

ax[0].scatter(X[:, 1], X[:, 0], s=5, color='lightblue')
ax[0].set_title(f'Geographic distribution ({len(X)} pincodes)')
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')

# colour by district to see natural structure before any algorithm runs
districts = pincode_df['District'].unique()
for i, dist in enumerate(districts):
    mask = (pincode_df['District'] == dist)
    ax[1].scatter(pincode_df.loc[mask, 'Longitude'], pincode_df.loc[mask, 'Latitude'],s=5,color=COLORS[i%len(COLORS)], label=dist)
ax[1].set_title('Colour by Revenue District')
ax[1].set_xlabel('Longitude')
ax[1].set_ylabel('Latitude')

plt.tight_layout()
plt.show()

In [ ]:
# Visualising on actual map if they lie inside physical boundary or not with geopandas
import geopandas as gpd
# https://github.com/guneetnarula/indian-district-boundaries/blob/master/topojson/state-wise/westbengal.json
gdf = gpd.read_file("westbengal.json")
fig,ax = plt.subplots(figsize=(6,6))
gdf.plot(ax=ax,color="gray",alpha=0.5)
for district in pincode_df['District'].unique():
    mask = pincode_df['District']==district
    ax.scatter(pincode_df["Longitude"][mask], pincode_df["Latitude"][mask], s=10,alpha=0.5,label=f"{district}")
ax.legend(bbox_to_anchor=(1,0.5),fontsize=5)
plt.tight_layout()
plt.show()

## 6. Algorithm Implementation

### 6.1 Helper functions
- K-Means++ Initialisation: instead of randomly init of centroids, each succesive centroid is chosen with probability proportional to its distance from the nearest existing centroid. Implemented as:
    1. Pick 1st centroid randomly from data points `X[rng.integers(n_samples)]`
    2. For remaining `k-1` centroids calculate squared euclidean distance from every point to nearest centroid
    3. Create a probability distribution where $P(x) \propto {D(x)^2}_{nearest-centr}$
    4. Select next centroid based on these weights, so distant points have higher chance of being picked

In [ ]:
def kpp_init(X,k,rng):
    n = X.shape[0]
    # random first centroid
    centroids = [X[rng.integers(n)]]
    # pick next k-1 based on algorithm descirbed above
    for _ in range(k-1):
        D = dist_matrix(X,np.array(centroids))
        min_dist = D.min(axis=1)
        w = (min_dist**2)
        probs = w/sum(w)
        centroids.append(X[rng.choice(n,p=probs)])

    return np.array(centroids)        

### 6.2 K-Means

**Convergence criterion**: stop when no label changes between iterations -- this is the true fixed point of the algorithm. Centroid-shift < tol is a proxy that can miss label oscillations near boundaries.

**Empty cluster handling**: if a cluster loses all its points, reinitialise its centroid to the point furthest from its current centroid -- not silently keeping the old one (which wastes a centroid and gives effectively K-1 clusters while reporting K).

**Multiple restarts**: K-Means is non-convex and each run can find a different local minimum. Running R times and keeping the best WCSS result.
* wscc - within cluster sum of squared distances

In [ ]:
def k_means(X,k,rng,max_iter=100):
    n = X.shape[0]
    centroids = kpp_init(X,k,rng)
    # since convergence criteria chosen is labels not changing we need to store prev labels
    labels = np.zeros(n)
    # wcss history for convergence analysis based on normal K-means objective - sum of sqaures of distance
    wcss_history = []
    for _ in range(max_iter):
        D = dist_matrix(X,np.array(centroids))
        # index for labels along each point
        new_labels = np.argmin(D,axis=1)
        wcss = (D[np.arange(n),new_labels]**2).sum()
        wcss_history.append(wcss)

        new_centroids = np.zeros_like(centroids)

        for i in range(k):
            pts = X[new_labels==i]
            if len(pts) > 0:
                new_centroids[i] = pts.mean(axis=0)
            else:
                # if empty cluster try initalising to diff centroid
                new_centroids[i] = X[D[:,i].argmax()]
        # if labels dont change break
        if np.array_equal(new_labels,labels):
            labels = new_labels
            centroids = new_centroids
            break
        labels = new_labels
        centroids = new_centroids
    return labels,centroids,wcss_history

def kmeans_best_of_r(X, k, r=10):
    # run r times, keep the result with lowest final WCSS
    results = []
    for i in range(r):
        labels, centroids, wcss_hist = k_means(X, k, np.random.default_rng(SEED + i))
        results.append((wcss_hist[-1], labels, centroids, wcss_hist))
    wcss_vals = np.array([res[0] for res in results])
    best = min(results, key=lambda x: x[0])
    return best[1], best[2], best[3], wcss_vals

### 6.3 K-Medoids + Fuzzy C-Means pipeline

**K-Medoids**: medoids are actual data points so the cluster centre can never lie in a physically infeasible region (a river, the sea). Minimises sum of absolute distances (not squared), which makes it less sensitive to outliers. Trade-off: O(n^2) per iteration so it scales poorly.

**Fuzzy C-Means initialised from medoid assignments**: instead of random FCM init, we convert K-Medoids hard labels into a soft membership matrix. This means FCM starts from a geometrically meaningful state -- the outlier-robust medoid partition -- and refines it into soft boundaries. We get both: robustness from medoids, boundary ambiguity analysis from FCM, in one pipeline.

FCM update rules:
$u_{ij} = \frac{1}{\sum_{l=1}^{k} \left(\frac{d_{ij}}{d_{il}}\right)^{\frac{2}{m-1}}}$
$c_j = \frac{\sum_i u_{ij}^m x_i}{\sum_i u_{ij}^m}$

FCM objective function is $\sum_{i=1}^{n} \sum_{j=1}^{c} u_{ij}^{m} ||x_i-c_j||^2$

Fuzziness $m=2$ is the standard choice. $m \to 1$ recovers hard K-Means, $m \to \infty$ makes all memberships equal $1/k$.

* referenced from pre read docs : https://en.wikipedia.org/wiki/Fuzzy_clustering

In [ ]:
def kmedoids(X,k,rng,max_iter=100):
    # init: pick k random points as medoid indices
    n = X.shape[0]
    medoid_idxs = rng.choice(n,k,replace=False)
    labels = np.zeros(n)
    # objective history for convergence analysis - sum of distances (not squared)
    obj_history = []

    for _ in range(max_iter):
        D = dist_matrix(X,X[medoid_idxs])
        new_labels = np.argmin(D,axis=1)
        obj = D[np.arange(n),new_labels].sum()
        obj_history.append(obj)
        improved = False
        for j in range(k):
            cluster_pts = np.where(new_labels==j)[0]
            if len(cluster_pts) == 0:
                continue
            # use the current medoid as best for comparison
            best_cost = D[cluster_pts,j].sum()
            best_med = medoid_idxs[j]
            for cand in cluster_pts:
                cost = dist_matrix(X[cluster_pts],X[[cand]])[:,0].sum()
                if cost < best_cost:
                    best_cost = cost
                    best_med = cand
                    improved = True

            medoid_idxs[j] = best_med
        if np.array_equal(new_labels,labels):
            labels = new_labels
            break
        labels = new_labels
        if not improved:
            break
    return labels,medoid_idxs,obj_history


def labels_to_membership(labels,k,sharpness=10.0):
    # convert hard labels to soft membership matrix for FCM init
    # assigned cluster gets high membership, others get low values
    n = len(labels)
    # create a membership matrix where i,jth component is membership of ith unit in jth label
    # initialise all to be equal and with moderate sharpness (between soft(2) and hard(50))
    U = np.ones((n,k))*(1.0/(sharpness*k))
    for i in range(n):
        # give the labels from kmedoids the highest membership as 1-(other mass)
        U[i,labels[i]] = 1.0 - (k-1)*(1.0/(sharpness*k))
    return U

def fuzzy_cmeans(X,U_init,m=2,max_iter=300,tol=1e-6):
    n = X.shape[0]
    U = U_init.copy()
    # objective history for convergence analysis - fuzzy c-means objective
    obj_history = []
    for _ in range(max_iter):
        # raise memberships to power m, large get amplified, small get suppressed
        Um = U**m
        # each centroid is a weighted averge, where weights are the fuzzy memberships
        centroids = (Um.T @ X) / Um.sum(axis=0)[:,None]

        D = dist_matrix(X,centroids)
        # avoid division by zero if a point lands exactly on centroid
        D = np.where(D==0,1e-10,D)
        # exponent obtained by minimising the objective
        exp = 2/(m-1)
        # build the Uij nd Uik from broadcasting
        # compare for each point distance to cluster j vs every other cluster k
        ratio = (D[:,:,None] / D[:,None,:])**exp
        # closer centroid -> smaller distance -> more membershi
        new_U = 1.0 / ratio.sum(axis=2)
        obj = (new_U**m * D**2).sum()
        obj_history.append(obj)

        if np.abs(new_U - U).max() < tol:
            U = new_U
            break
        U = new_U

    labels = np.argmax(U,axis=1)
    return labels,centroids,U,obj_history

### 6.4 Silhouette Score

Computed with equirectangular distance -- must match the clustering metric.

$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$

where $a(i)$ = mean intra-cluster distance, $b(i)$ = mean distance to nearest other cluster.

Silhouette measures compactness within cluster.

In [ ]:
def silhouette(X, labels):
    unique = np.unique(labels)
    scores = np.zeros(len(X))
    for i in range(len(X)):
        ci = labels[i]
        same = X[labels == ci]

        # a(i): vectorised distance to all same-cluster points
        if len(same) > 1:
            diff = same - X[i]
            diff[:, 1] *= COS_LAT
            d_same = np.sqrt((diff**2).sum(axis=1))
            a = d_same[d_same > 0].mean()
        else:
            a = 0.0
        # b(i): mean distance to each other cluster, take minimum
        b_vals = []
        for cj in unique:
            if cj == ci:
                continue
            other = X[labels == cj]
            diff = other - X[i]
            diff[:, 1] *= COS_LAT
            b_vals.append(np.sqrt((diff**2).sum(axis=1)).mean())
        b = min(b_vals) if b_vals else 0.0
        m = max(a, b)
        scores[i] = (b - a) / m if m > 0 else 0.0

    return scores.mean(), scores

## 7. K Selection

Domain priors: 23 revenue districts, 28 postal divisions.
Candidates chosen to cover coarse (5), mid (8, 12), and division-aligned (23) regions.

Elbow: plot percentage WCSS reduction per additional cluster,
Raw WCSS decreases monotonically by construction -- the rate of decrease is the actual signal.

In [ ]:
K_CANDIDATES = [5, 8, 12, 16, 23]
k_results = {}

for k in K_CANDIDATES:
    labels_k, cents_k, wcss_hist_k, wcss_runs = kmeans_best_of_r(X, k, r=10)
    sil_k, _ = silhouette(X, labels_k)
    k_results[k] = {'labels': labels_k, 'centroids': cents_k, 'wcss': wcss_hist_k[-1], 'wcss_runs': wcss_runs, 'silhouette': sil_k}
    print(f"K={k:2d} | WCSS={wcss_hist_k[-1]:.6f} | Sil={sil_k:.4f}")

In [ ]:
k_vals = list(k_results.keys())
wcss_vals = [k_results[k]['wcss'] for k in k_vals]
sil_vals = [k_results[k]['silhouette'] for k in k_vals]

# percentage WCSS reduction to see  where the rate of improvement drops off
pct_reduction = [(wcss_vals[i] - wcss_vals[i+1])/wcss_vals[i] * 100 for i in range(len(wcss_vals)-1)]

fig, ax = plt.subplots(1,3,figsize=(12,6))
fig.suptitle('K Selection')

ax[0].plot(k_vals, wcss_vals, 'o-', color='red')
ax[0].set_title('Raw WCSS vs K')
ax[0].set_xlabel('K')
ax[0].set_ylabel('WCSS (degrees sq)')
ax[0].grid(True)

ax[1].plot(k_vals[:-1], pct_reduction, 'o-', color='orange')
ax[1].set_title('% WCSS reduction per additional K')
ax[1].set_xlabel('K')
ax[1].set_ylabel('% reduction')
ax[1].grid(True)

ax[2].plot(k_vals, sil_vals, 'o-', color='blue')
ax[2].set_title('Silhouette vs K')
ax[2].set_xlabel('K')
ax[2].set_ylabel('Silhouette score')
ax[2].grid(True)

plt.tight_layout()
plt.show()

### K decision

- The clustering metrics do not yield a sharp optimal K. Silhouette favors smaller K (~5), while WCSS decreases smoothly without a clear elbow. 
- Therefore I selected K=8 as it provides ~2-3 districts per cluster which can be useful for regional segmentation

In [ ]:
K = 8
labels_km, centroids_km, wcss_hist_km, _ =kmeans_best_of_r(X, K, r=10)

# K-Medoids + FCM
labels_kmed, medoid_idxs, obj_hist_kmed=kmedoids(X, K, np.random.default_rng(SEED))
U_init=labels_to_membership(labels_kmed, K)  
labels_fcm, centroids_fcm, U_fcm, obj_hist_fcm=fuzzy_cmeans(X, U_init)

# attach labels to dataframe
pincode_df['km_cluster'] =labels_km
pincode_df['kmed_cluster'] =labels_kmed
pincode_df['fcm_cluster'] =labels_fcm

sil_km, _ =silhouette(X,labels_km)
sil_kmed, _ =silhouette(X,labels_kmed)
sil_fcm, _ =silhouette(X,labels_fcm)

print(f"K-Means   | Silhouette: {sil_km:.4f}")
print(f"K-Medoids | Silhouette: {sil_kmed:.4f}")
print(f"Fuzzy CM  | Silhouette: {sil_fcm:.4f}")

# cluster size ratio > 5 = density imbalance driving clustering not geography
counts = np.bincount(labels_km.astype(int))
ratio = counts.max()/counts.min()
print(f"Cluster sizes: {np.sort(counts)}")
print(f"Max/min ratio: {ratio:.2f}")

## 8. Visualisations

### 8.1 K-Means vs K-Medoids vs Fuzzy CM

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'West Bengal Pincode Clustering (K={K})')

for k in range(K):
    mask = labels_km == k
    ax[0].scatter(X[mask, 1], X[mask, 0], s=8, alpha=0.6, color=COLORS[k], label=f'C{k+1}')
ax[0].scatter(centroids_km[:, 1], centroids_km[:, 0], marker='*', s=200, color='black')
ax[0].set_title('K-Means')

for k in range(K):
    mask = labels_kmed == k
    ax[1].scatter(X[mask, 1], X[mask, 0], s=8, alpha=0.6, color=COLORS[k], label=f'C{k+1}')
ax[1].scatter(X[medoid_idxs, 1], X[medoid_idxs, 0], marker='D', s=100, color='black')
ax[1].set_title('K-Medoids')

for k in range(K):
    mask = labels_fcm == k
    ax[2].scatter(X[mask, 1], X[mask, 0], s=8, alpha=0.6, color=COLORS[k], label=f'C{k+1}')
ax[2].scatter(centroids_fcm[:, 1], centroids_fcm[:, 0], marker='*', s=200, color='black')
ax[2].set_title('Fuzzy CM (init from medoids)')

for a in ax:
    a.set_xlabel('Longitude')
    a.set_ylabel('Latitude')
    a.legend(fontsize=6)
    a.grid(True)

plt.tight_layout()
plt.show()

### 8.2 K-Means vs actual postal divisions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 7))

for k in range(K):
    mask = labels_km == k
    ax[0].scatter(X[mask, 1], X[mask, 0], s=8, alpha=0.6, color=COLORS[k], label=f'C{k+1}')
ax[0].scatter(centroids_km[:, 1], centroids_km[:, 0], marker='*', s=200, color='black')
ax[0].set_title('K-Means Clusters')
ax[0].legend()
divisions = pincode_df['DivisionName'].fillna('Unknown').unique()
for i, div in enumerate(divisions):
    mask = pincode_df['DivisionName'].fillna('Unknown') == div
    ax[1].scatter(pincode_df.loc[mask, 'Longitude'], pincode_df.loc[mask, 'Latitude'], s=8, alpha=0.6, color=COLORS[i % len(COLORS)], label=div)
ax[1].set_title('Actual Postal Divisions')
ax[1].legend(fontsize=5,bbox_to_anchor=(1, 0.5))
for a in ax:
    a.set_xlabel('Longitude')
    a.set_ylabel('Latitude')
    a.grid(True)

plt.tight_layout()
plt.show()

### 8.3 Convergence

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12, 4))

ax[0].plot(wcss_hist_km, 'o-', color='red')
ax[0].set_title('K-Means Objective (WCSS)')
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('WCSS')
ax[0].grid(True)

ax[1].plot(obj_hist_kmed, 'o-', color='blue')
ax[1].set_title('K-Medoids Objective (sum distances)')
ax[1].set_xlabel('Iteration')
ax[1].set_ylabel('Sum of distances')
ax[1].grid(True)

plt.tight_layout()
plt.show()

### 8.4 Fuzzy membership -- boundary ambiguity

Points with high max-membership (close to 1.0) are clearly in one cluster. Points near 1/K are boundary-ambiguous -- K-means would hard-assign them arbitrarily.

In [ ]:
max_membership = U_fcm.max(axis=1)  # how sure FCM is about each point
low = np.quantile(max_membership, 0.25)
high = np.quantile(max_membership, 0.75)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# colour points by certainty - green=certain, red=ambiguous
# bin membership into ranges and assign colors manually
def membership_color(val):
    if val > high:
        return 'green'
    elif val > low:
        return 'orange'
    else:
        return 'red'

point_colors = [membership_color(v) for v in max_membership]
ax[0].scatter(X[:, 1], X[:, 0], color=point_colors, s=10, alpha=0.7)
# manual legend on top 
ax[0].scatter([], [], color='green', label=f'>{high} certain')
ax[0].scatter([], [], color='orange', label=f'{low}-{high}')
ax[0].scatter([], [], color='red', label=f'<{low} ambiguous')
ax[0].set_title('Fuzzy membership certainty')
ax[0].set_xlabel('Longitude')
ax[0].set_ylabel('Latitude')
ax[0].legend()
ax[0].grid(True)

ax[1].hist(max_membership, bins=30, color='steelblue')
ax[1].set_title('Distribution of max membership')
ax[1].set_xlabel('Max membership')
ax[1].set_ylabel('Count')
ax[1].grid(True)

plt.tight_layout()
plt.show()

ambiguous = (max_membership < low).sum()
print(f"Pincodes with max membership < {low:.2f} (ambiguous): {ambiguous} / {len(X)}")

## 9. Cluster-level analysis

In [ ]:
stats = pincode_df.groupby('km_cluster').agg(
    n_pincodes=('Pincode', 'count'),
    n_districts=('District', 'nunique'),
    n_offices=('n_offices', 'sum'),
    lat_centre=('Latitude', 'mean'),
    lon_centre=('Longitude', 'mean'),
    lat_std=('Latitude', 'std'),
    lon_std=('Longitude', 'std'),
)

# spread in km
stats['spread_km'] = np.sqrt((stats['lat_std']*111)**2 + (stats['lon_std']*111*COS_LAT)**2)
# offices/pincode: proxy for postal density -> proxy for urbanisation
stats['offices_per_pincode'] = (stats['n_offices'] / stats['n_pincodes']).round(2)

print("K-Means Cluster Statistics:")
print(stats.round(3).to_string())

In [ ]:
labels_x = [f'C{k+1}' for k in range(K)]

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Cluster Characteristics')

ax[0].bar(labels_x, stats['n_pincodes'], color=COLORS[:K])
ax[0].set_title('Pincodes per cluster')
ax[0].set_ylabel('Count')

ax[1].bar(labels_x, stats['spread_km'].round(1), color=COLORS[:K])
ax[1].set_title('Geographic spread (km)')
ax[1].set_ylabel('Std deviation km')

ax[2].bar(labels_x, stats['offices_per_pincode'], color=COLORS[:K])
ax[2].set_title('Post offices per pincode')
ax[2].set_ylabel('Ratio')

plt.tight_layout()
plt.show()

In [ ]:
# disagreements between K-Means and K-Medoids show boundary-ambiguous pincodes
disagree = (labels_km != labels_kmed).sum()
print(f"K-Means vs K-Medoids disagreements: {disagree}/{len(X)} pincodes")
print(f"Agreement rate: {(1-disagree/len(X))*100:.1f}%")

print(f"\nSilhouette scores:")
print(f"  K-Means: {sil_km:.4f}")
print(f"  K-Medoids: {sil_kmed:.4f}")
print(f"  Fuzzy CM: {sil_fcm:.4f}")

# K-Medoids medoid offices - actual real post offices at cluster centres
print(f"\nK-Medoids hub offices:")
for j, idx in enumerate(medoid_idxs):
    row = pincode_df.iloc[idx]
    print(f"  C{j+1}: {row['OfficeName']}  ({row['Latitude']:.4f}N, {row['Longitude']:.4f}E)  [{row['District']}]")

## 10. Inferences

**Read from cluster statistics, not assumed beforehand**

**Urban-rural gradient:**
Cluster with highest `offices_per_pincode` corresponds to Kolkata and inner suburbs. Each pincode in the metro is served by multiple sub-offices and branch offices -- the postal network is fine-grained. Compare to northern/western clusters where one pincode often has only a single head office. This directly reflects population density and settlement patterns.

**Geographic spread as a coverage metric:**
Clusters with high `spread_km` are geographically large but have fewer pincodes. This means individual pincodes cover more physical territory -- each post office is responsible for a wider area. In northern area + norther belt sparse coverage is due to the high altitude / entering north-eastern India domain with low population

**Ambiguous areas:**
K-Means and K-Medoids disagreements concentrate at cluster boundaries. Pincodes where algorithms disagree are genuinely boundary cases -- their geographic position makes cluster assignment ambiguous. The FCM membership plot makes this explicit: low-certainty pincodes near administrative boundaries are the regions where any hard clustering would draw an arbitrary line.

**Division alignment:**
Comparing K=8 clusters to actual postal divisions -- matches confirm boundaries follow geographic proximity. Mismatches show where postal admin structure deviates from geography, likely due to historical routing or river barriers.